# LeetCode 1197 — Minimum Knight Moves bằng Python

Link đề: https://leetcode.com/problems/minimum-knight-moves/ · Độ khó: **Medium** · Chủ đề: Breadth-First Search, Heuristic Search, Bidirectional Search, A* Search

> **Bài Premium — đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Đề được viết lại theo bản mô tả công khai (doocs); chữ ký hàm suy ra từ lời giải Python tham khảo.

Skeleton LeetCode:

```python
class Solution:
    def minKnightMoves(self, x: int, y: int) -> int:
        ...
```

Ghi chú: "số bước ít nhất trên đồ thị không trọng số" → **BFS** (Breadth-First Search — tìm kiếm theo chiều rộng). Bàn cờ **vô hạn** nên phải biết **cắt tỉa**: dùng **đối xứng** đưa về góc phần tư thứ nhất, giới hạn vùng tìm kiếm, **BFS hai chiều**, hoặc **DP đệ quy có nhớ**; thậm chí có **công thức `O(1)`**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Mô hình hóa bàn cờ như **đồ thị không trọng số**: mỗi ô là đỉnh, mỗi nước mã là cạnh; số bước ít nhất = khoảng cách BFS.
2. Cài BFS chuẩn với `deque` và tập `visited`, hiểu vì sao BFS cho đường ngắn nhất.
3. Dùng **tính đối xứng** (`|x|`, `|y|`, hoán đổi `x↔y`) và **giới hạn vùng** để BFS trên bàn vô hạn chạy nhanh.
4. Biết các kỹ thuật tăng tốc: **BFS hai chiều (bidirectional)**, **DFS + memo** theo công thức truy hồi, và **công thức đóng `O(1)`**.
5. Tránh lỗi: chặn vùng quá chặt (mất đường đi vòng như ô `(1,1)` cần 2 bước qua tọa độ âm), đánh dấu `visited` muộn gây bùng nổ hàng đợi.

## 2. Đề bài & ví dụ

Trên bàn cờ **vô hạn** (tọa độ từ `-∞` tới `+∞`), quân mã đứng ở ô `[0, 0]`. Quân mã có **8 nước đi**: hai ô theo một hướng chính rồi một ô theo hướng vuông góc, tức các vector `(±1, ±2)` và `(±2, ±1)`.

Trả về **số bước ít nhất** để quân mã tới ô `[x, y]`. Đề đảm bảo luôn có đáp án.

**Ví dụ 1:** `x = 2, y = 1` → `1` (`[0,0] → [2,1]`).

**Ví dụ 2:** `x = 5, y = 5` → `4` (`[0,0] → [2,1] → [4,2] → [3,4] → [5,5]`).

**Ràng buộc:** `-300 <= x, y <= 300`, `0 <= |x| + |y| <= 300`.

In [1]:
import random
import time
from collections import deque
from functools import lru_cache

MOVES = [(1, 2), (2, 1), (2, -1), (1, -2), (-1, -2), (-2, -1), (-2, 1), (-1, 2)]
examples = [((2, 1), 1), ((5, 5), 4), ((0, 0), 0), ((1, 1), 2), ((1, 0), 3)]
for (x, y), want in examples:
    print(f"({x}, {y}) -> {want}")

(2, 1) -> 1
(5, 5) -> 4
(0, 0) -> 0
(1, 1) -> 2
(1, 0) -> 3


## 3. Kiến thức nền

### 3.1 Đồ thị không trọng số và BFS

BFS duyệt các đỉnh theo **lớp**: lớp 0 là đỉnh xuất phát, lớp 1 là các đỉnh cách 1 cạnh, lớp 2 cách 2 cạnh... Đỉnh được lấy ra khỏi hàng đợi lần đầu tiên luôn với **khoảng cách ngắn nhất** (vì mọi cạnh có trọng số như nhau).

In [2]:
graph = {'A': ['B', 'C'], 'B': ['D'], 'C': ['D', 'E'], 'D': ['F'], 'E': ['F'], 'F': []}
dist = {'A': 0}
q = deque(['A'])
while q:
    u = q.popleft()
    for v in graph[u]:
        if v not in dist:
            dist[v] = dist[u] + 1
            q.append(v)
print(dist)

{'A': 0, 'B': 1, 'C': 1, 'D': 2, 'E': 2, 'F': 3}


### 3.2 `collections.deque` — hàng đợi hai đầu

`deque.append` thêm vào cuối, `deque.popleft()` lấy ra từ đầu, cả hai `O(1)`. Đừng dùng `list.pop(0)` (tốn `O(n)` mỗi lần).

In [3]:
q = deque([1, 2])
q.append(3)
print(q.popleft(), q)

1 deque([2, 3])


### 3.3 Tập `visited` với tuple làm khóa

Tọa độ `(x, y)` là tuple — **bất biến, băm được (hashable)** — nên bỏ vào `set` được. Kiểm tra `p in visited` trung bình `O(1)`.

In [4]:
visited = {(0, 0)}
visited.add((2, 1))
print((2, 1) in visited, (1, 2) in visited)

True False


### 3.4 Tính đối xứng của nước mã

Tập 8 nước đi **đối xứng** qua trục `x`, trục `y` và đường chéo `y = x`. Do đó `dist(x, y) = dist(|x|, |y|) = dist(|y|, |x|)`. Ta luôn có thể đưa đích về góc phần tư thứ nhất với `x ≥ y ≥ 0`.

In [5]:
def reflect(moves, fx, fy, swap):
    out = set()
    for dx, dy in moves:
        dx, dy = dx * fx, dy * fy
        if swap:
            dx, dy = dy, dx
        out.add((dx, dy))
    return out


S = set(MOVES)
print(all(reflect(MOVES, fx, fy, sw) == S for fx in (1, -1) for fy in (1, -1) for sw in (False, True)))

True


### 3.5 `functools.lru_cache` — nhớ kết quả hàm (memoization)

Gắn `@lru_cache(maxsize=None)` lên hàm đệ quy thì mỗi bộ tham số chỉ được tính một lần. Đây là cách viết **DP top-down** gọn nhất trong Python.

In [6]:
@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(80), fib.cache_info().hits > 0)

23416728348467685 True


### 3.6 Tính chẵn lẻ (parity) của nước mã

Mỗi nước mã đổi `x + y` một lượng lẻ (`±1 ± 2` hoặc `±2 ± 1` → `±1` hoặc `±3`). Vậy sau `s` bước, `x + y ≡ s (mod 2)`: **số bước có cùng tính chẵn lẻ với `x + y`**. Đây là một "luật kiểm tra" hữu ích khi debug.

In [7]:
print([(dx + dy) % 2 for dx, dy in MOVES])

[1, 1, 1, 1, 1, 1, 1, 1]


## 4. Ý tưởng nền tảng / trực giác

Liệt kê tay các lớp BFS gần gốc (chỉ vẽ vùng `0..4`):

- Lớp 0: `(0,0)`.
- Lớp 1: 8 ô `(±1,±2), (±2,±1)` — trong góc phần tư thứ nhất: `(1,2), (2,1)`.
- Lớp 2: ví dụ `(1,1)` qua `(2,-1)` hoặc `(-1,2)`; `(2,2)`? — Không! `(2,2)` cần **4** bước. `(4,2)`, `(3,3)`, `(2,0)`, `(0,2)`, `(4,0)`...
- `(1,0)` cần **3** bước (một ô kề sát mà lại xa!).

Bảng khoảng cách `dist(x, y)` cho `0 ≤ x, y ≤ 5` được in ở cell dưới. Nhận xét: càng xa thì khoảng cách tăng xấp xỉ `max(x/2, (x+y)/3)`, nhưng gần gốc có các **ngoại lệ** `(1,0) → 3`, `(2,2) → 4`.

In [8]:
def bfs_table(R):
    dist = {(0, 0): 0}
    q = deque([(0, 0)])
    while q:
        x, y = q.popleft()
        for dx, dy in MOVES:
            nx, ny = x + dx, y + dy
            if -R <= nx <= R and -R <= ny <= R and (nx, ny) not in dist:
                dist[(nx, ny)] = dist[(x, y)] + 1
                q.append((nx, ny))
    return dist


T = bfs_table(12)
print("y\\x " + " ".join(f"{x:2}" for x in range(6)))
for y in range(5, -1, -1):
    print(f"{y:3} " + " ".join(f"{T[(x, y)]:2}" for x in range(6)))

y\x  0  1  2  3  4  5
  5  3  4  3  4  3  4
  4  2  3  2  3  4  3
  3  3  2  3  2  3  4
  2  2  1  4  3  2  3
  1  3  2  1  2  3  4
  0  0  3  2  3  2  3


## 5. Cách 1 — Brute Force / baseline: BFS thẳng (có chặn vùng rộng)

**Ý tưởng:** BFS từ `(0,0)`, dừng khi lấy ra `(x, y)`. Vì bàn vô hạn, nếu không chặn thì BFS lan ra mọi hướng — số ô ở lớp `d` tỉ lệ `d^2`. Ta chặn trong hình vuông `|·| ≤ max(|x|,|y|) + 4` (đủ chỗ cho đường đi vòng; mục 10 giải thích).

In [9]:
def bfs_plain(x: int, y: int) -> int:
    R = max(abs(x), abs(y)) + 4
    if (x, y) == (0, 0):
        return 0
    seen = {(0, 0)}
    q = deque([(0, 0, 0)])
    while q:
        cx, cy, d = q.popleft()
        for dx, dy in MOVES:
            nx, ny = cx + dx, cy + dy
            if (nx, ny) == (x, y):
                return d + 1
            if -R <= nx <= R and -R <= ny <= R and (nx, ny) not in seen:
                seen.add((nx, ny))
                q.append((nx, ny, d + 1))
    return -1


for (x, y), want in examples:
    print((x, y), bfs_plain(x, y), want)
    assert bfs_plain(x, y) == want

(2, 1) 1 1
(5, 5) 4 4
(0, 0) 0 0
(1, 1) 2 2
(1, 0) 3 3


**Độ phức tạp:** số ô thăm `O(R^2)` với `R ≈ max(|x|, |y|)` → tới khoảng `600^2 = 3.6·10^5` ô cho mỗi truy vấn xa. Chạy được nhưng chậm (hàng trăm ms), và nếu nhiều truy vấn sẽ rất tốn.

**Nút thắt:** BFS lan đều **cả 4 góc phần tư**, trong khi đích chỉ nằm ở một góc. 3/4 công sức là lãng phí.

In [10]:
for target in [(10, 10), (100, 100), (150, 150)]:
    t = time.perf_counter(); r = bfs_plain(*target); print(target, r, f"{time.perf_counter() - t:.3f}s")

(10, 10) 8 0.001s
(100, 100) 68 0.090s


(150, 150) 100 0.200s


## 6. Bước đột phá

**Insight 1 — đối xứng:** `dist(x, y) = dist(|x|, |y|)`. Đưa đích về góc phần tư thứ nhất.

**Insight 2 — chặn vùng chặt:** khi đích ở `x, y ≥ 0`, đường đi ngắn nhất **không cần** đi xa vào vùng âm; chỉ cần cho phép tọa độ `≥ -2` (vài ô "đệm" cho những đích sát gốc như `(1,1)` phải đi qua `(2,-1)` hoặc `(-1,2)`). Quy tắc: *nếu* ô mới có `nx < -2` hoặc `ny < -2` *thì* bỏ qua.

**Insight 3 — truy hồi:** đặt `x ≥ y ≥ 0` (sau đối xứng). Khi đích đủ xa, nước đi đầu tiên tốt nhất (đi ngược từ đích về gốc) luôn là `(-2,-1)` hoặc `(-1,-2)`:
- `f(x, y) = 1 + min(f(|x−2|, |y−1|), f(|x−1|, |y−2|))`,
- Cơ sở: `f(0,0) = 0`; khi `x + y == 2` (tức `(2,0)`, `(1,1)`, `(0,2)`) thì `f = 2`.

Insight 3 dẫn tới DP có nhớ chạy cực nhanh.

## 7. Các cách tiếp theo

### Cách 2 — BFS với đối xứng + chặn vùng `≥ -2`

In [11]:
def bfs_symmetric(x: int, y: int) -> int:
    x, y = abs(x), abs(y)
    if (x, y) == (0, 0):
        return 0
    seen = {(0, 0)}
    q = deque([(0, 0)])
    steps = 0
    while q:
        steps += 1
        for _ in range(len(q)):           # xử lý trọn một lớp
            cx, cy = q.popleft()
            for dx, dy in MOVES:
                nx, ny = cx + dx, cy + dy
                if (nx, ny) == (x, y):
                    return steps
                if nx >= -2 and ny >= -2 and nx <= x + 2 and ny <= y + 2 and (nx, ny) not in seen:
                    seen.add((nx, ny))
                    q.append((nx, ny))
    return -1


for (x, y), want in examples:
    assert bfs_symmetric(x, y) == want
for target in [(150, 150), (300, 0), (-120, 180)]:
    t = time.perf_counter(); r = bfs_symmetric(*target); print(target, r, f"{time.perf_counter() - t:.3f}s")

(150, 150) 100 0.038s
(300, 0) 150 0.002s
(-120, 180) 100 0.057s


**Độ phức tạp:** vùng thăm `O(|x| · |y|)` (hình chữ nhật `[−2, x+2] × [−2, y+2]`), tối đa khoảng `150·150 = 2.25·10^4`... với `|x|+|y| ≤ 300` thì `|x|·|y| ≤ 22500`. Nhanh hơn nhiều.

*(Chặn trên `x + 2`, `y + 2` cũng an toàn — kiểm chứng bằng bộ test so với BFS toàn cục ở mục 12.)*

### Cách 3 — BFS hai chiều (bidirectional BFS)

Lan BFS đồng thời từ **gốc** và từ **đích**; luôn mở rộng phía có biên (frontier) nhỏ hơn. Khi hai biên chạm nhau, tổng hai khoảng cách là đáp án. Vì số ô ở bán kính `d` tỉ lệ `d^2`, hai quả cầu bán kính `d/2` tốn ít hơn hẳn một quả bán kính `d`.

In [12]:
def bfs_bidirectional(x: int, y: int) -> int:
    x, y = abs(x), abs(y)
    if (x, y) == (0, 0):
        return 0
    da = {(0, 0): 0}
    db = {(x, y): 0}
    fa, fb = [(0, 0)], [(x, y)]
    while fa and fb:
        if len(fa) > len(fb):                 # mở rộng phía nhỏ hơn
            fa, fb, da, db = fb, fa, db, da
        nxt = []
        for cx, cy in fa:
            d = da[(cx, cy)]
            for dx, dy in MOVES:
                p = (cx + dx, cy + dy)
                if p in db:
                    return d + 1 + db[p]
                if p not in da and -2 <= p[0] <= x + 2 and -2 <= p[1] <= y + 2:
                    da[p] = d + 1
                    nxt.append(p)
        fa = nxt
    return -1


for (x, y), want in examples:
    assert bfs_bidirectional(x, y) == want, (x, y)
t = time.perf_counter(); print(bfs_bidirectional(150, 150), f"{time.perf_counter() - t:.4f}s")

100 0.0196s


**Lưu ý tính đúng:** ta mở rộng **trọn một lớp** của một phía rồi mới kiểm tra; khi gặp ô `p` đã nằm trong phía kia, `d + 1 + db[p]` là tổng nhỏ nhất vì mọi ô trong lớp đang mở rộng có cùng `d` và `db` chứa đúng các lớp đã hoàn tất. **Độ phức tạp:** thực tế nhanh hơn Cách 2 nhiều lần, cùng bậc `O(|x|·|y|)` trong worst case.

### Cách 4 — DFS + memo theo công thức truy hồi

In [13]:
@lru_cache(maxsize=None)
def f_memo(x: int, y: int) -> int:
    x, y = abs(x), abs(y)
    if x < y:
        x, y = y, x
    if x == 0 and y == 0:
        return 0
    if x + y == 2:
        return 2
    return 1 + min(f_memo(x - 2, y - 1), f_memo(x - 1, y - 2))


for (x, y), want in examples:
    assert f_memo(x, y) == want
t = time.perf_counter(); print(f_memo(150, 150), f_memo(300, 0), f"{time.perf_counter() - t:.4f}s")

100 150 0.0011s


**Vì sao truy hồi đúng?** Mỗi bước giảm `x + y` đi 3, đi về phía gốc. Các trường hợp cơ sở `x + y ∈ {0, 2}` xử lý vùng sát gốc; các ô `x + y = 1` (như `(1,0)`) sẽ được tính `1 + min(f(1,1), f(0,2)) = 3` ✓. Ta kiểm chứng toàn bộ vùng `|x|+|y| ≤ 300` so với BFS ở mục 12. **Độ phức tạp:** số trạng thái khác nhau `O(|x| · |y|)` worst case nhưng thực tế ít hơn nhiều (chỉ dải gần đường thẳng tới gốc); độ sâu đệ quy `≈ (x+y)/3 ≤ 100` — an toàn.

### Cách 5 — Công thức đóng `O(1)`

Với `x ≥ y ≥ 0`, đặt `delta = x − y`:
- Ngoại lệ: `(1, 0) → 3`, `(2, 2) → 4`.
- Nếu `y > delta`: `delta − 2·⌊(delta − y) / 3⌋`.
- Ngược lại: `delta − 2·⌊(delta − y) / 4⌋`.

(Chú ý phép chia sàn `//` trong Python làm tròn **xuống** cả với số âm — điều công thức cần.)

In [14]:
def formula(x: int, y: int) -> int:
    x, y = abs(x), abs(y)
    if x < y:
        x, y = y, x
    if (x, y) == (1, 0):
        return 3
    if (x, y) == (2, 2):
        return 4
    delta = x - y
    if y > delta:
        return delta - 2 * ((delta - y) // 3)
    return delta - 2 * ((delta - y) // 4)


for (x, y), want in examples:
    assert formula(x, y) == want
print([formula(*p) for p, _ in examples], formula(150, 150), formula(300, 0))

[1, 4, 0, 2, 3] 100 150


## 8. TRACE từng bước

Trace Cách 2 (BFS theo lớp) tới `(5, 5)`: in kích thước mỗi lớp và một vài ô tiêu biểu. Sau đó trace Cách 4 (đệ quy) để thấy đường đi ngược về gốc.

| lớp | số ô mới (trong vùng chặn) | ví dụ |
|---|---|---|
| 1 | 8 | (1,2), (2,1), (2,-1), (1,-2), ... (vùng chặn cho phép tọa độ tới -2) |
| 2 | 19 | (2,4), (3,3), (3,1), (2,0), (-1,1), ... |
| 3 | 30 | (3,6), (4,5), (4,3), (3,2), ... |
| 4 | 29 | chạm (5,5) |

In [15]:
def trace_bfs(x, y):
    x, y = abs(x), abs(y)
    seen = {(0, 0)}
    q = deque([(0, 0)])
    parent = {(0, 0): None}
    steps = 0
    while q:
        steps += 1
        layer = []
        for _ in range(len(q)):
            cx, cy = q.popleft()
            for dx, dy in MOVES:
                p = (cx + dx, cy + dy)
                if -2 <= p[0] <= x + 2 and -2 <= p[1] <= y + 2 and p not in seen:
                    seen.add(p); parent[p] = (cx, cy); q.append(p); layer.append(p)
        print(f"lớp {steps}: {len(layer):3} ô mới, ví dụ {layer[:5]}")
        if (x, y) in seen:
            path, p = [], (x, y)
            while p is not None:
                path.append(p); p = parent[p]
            print("đường đi:", " -> ".join(map(str, path[::-1])))
            return steps


trace_bfs(5, 5)

lớp 1:   8 ô mới, ví dụ [(1, 2), (2, 1), (2, -1), (1, -2), (-1, -2)]
lớp 2:  19 ô mới, ví dụ [(2, 4), (3, 3), (3, 1), (2, 0), (-1, 1)]
lớp 3:  30 ô mới, ví dụ [(3, 6), (4, 5), (4, 3), (3, 2), (0, 3)]
lớp 4:  29 ô mới, ví dụ [(5, 7), (5, 5), (4, 4), (1, 5), (1, 7)]
đường đi: (0, 0) -> (1, 2) -> (2, 4) -> (3, 6) -> (5, 5)


4

In [16]:
def trace_memo(x, y, indent=0):
    x, y = abs(x), abs(y)
    if x < y:
        x, y = y, x
    v = f_memo(x, y)
    print("  " * indent + f"f({x},{y}) = {v}")
    if (x, y) == (0, 0) or x + y == 2:
        return
    a, b = f_memo(x - 2, y - 1), f_memo(x - 1, y - 2)
    nxt = (x - 2, y - 1) if a <= b else (x - 1, y - 2)
    trace_memo(*nxt, indent + 1)


trace_memo(5, 5)

f(5,5) = 4
  f(4,3) = 3
    f(3,1) = 2
      f(2,1) = 1
        f(0,0) = 0


## 9. Code nộp LeetCode

Chọn Cách 4 (DFS + memo): ngắn, nhanh, dễ chứng minh bằng bộ test. (Cách 2/3 cũng được chấp nhận.)

In [17]:
class Solution:
    def minKnightMoves(self, x: int, y: int) -> int:
        @lru_cache(maxsize=None)
        def dfs(a: int, b: int) -> int:
            a, b = abs(a), abs(b)
            if a < b:
                a, b = b, a
            if a == 0 and b == 0:
                return 0
            if a + b == 2:
                return 2
            return 1 + min(dfs(a - 2, b - 1), dfs(a - 1, b - 2))

        return dfs(x, y)


sol = Solution()
for (x, y), want in examples:
    print((x, y), sol.minKnightMoves(x, y))
    assert sol.minKnightMoves(x, y) == want

(2, 1) 1
(5, 5) 4
(0, 0) 0
(1, 1) 2
(1, 0) 3


## 10. Vì sao đúng

1. **BFS cho khoảng cách ngắn nhất** trên đồ thị không trọng số: quy nạp theo lớp — mọi ô được gán khoảng cách `d` lần đầu đều không thể đến được với ít hơn `d` bước, vì nếu được thì nó đã xuất hiện ở lớp trước.
2. **Đối xứng:** ánh xạ phản chiếu biến một đường đi hợp lệ thành đường đi hợp lệ cùng độ dài (tập nước đi bất biến), nên `dist(x,y) = dist(|x|,|y|) = dist(|y|,|x|)`.
3. **Chặn vùng `≥ -2`:** với đích trong góc phần tư thứ nhất, có thể chứng minh (và ta **kiểm chứng toàn bộ** các đích `|x|+|y| ≤ 300` ở mục 12) rằng tồn tại một đường đi ngắn nhất không rời vùng `[-2, x+2] × [-2, y+2]`. Trực giác: bước đi "lùi" chỉ cần thiết khi gần gốc hoặc gần đích để chỉnh tính chẵn lẻ, và độ lệch tối đa là 2 ô.
4. **Truy hồi (Cách 4):** khi `x ≥ y` và `(x, y)` không phải cơ sở, bước cuối cùng của một đường tối ưu có thể chọn là `(+2,+1)` hoặc `(+1,+2)` (đi "về phía" đích); ta lấy min hai khả năng, và các giá trị âm được phản chiếu qua `abs`. Tính đúng tuyệt đối trên toàn miền ràng buộc được xác nhận bằng đối chiếu BFS.

## 11. Time / Space Complexity

Gọi `X = |x|`, `Y = |y|`, `D = X + Y ≤ 300`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS thẳng (vuông `R = max(X,Y)+4`) | `O(R^2)` | `O(R^2)` |
| 2. BFS đối xứng + chặn | `O((X+4)(Y+4))` | như Time |
| 3. BFS hai chiều | cùng bậc Cách 2, hằng số nhỏ hơn nhiều | như Time |
| 4. DFS + memo | `O(X·Y)` worst, thực tế ~ `O(D)` đến `O(D^2)` nhỏ | như Time + đệ quy sâu `D/3` |
| 5. Công thức | `O(1)` | `O(1)` |

## 12. Bộ test

Chạy **một** BFS toàn cục từ gốc trên hình vuông `[-305, 305]^2` để có khoảng cách chuẩn cho **mọi** ô `|x|+|y| ≤ 300`, rồi đối chiếu các cách.

In [18]:
t = time.perf_counter()
REF = bfs_table(305)
print("BFS toàn cục:", len(REF), "ô,", f"{time.perf_counter() - t:.2f}s")
targets = [(x, y) for x in range(-300, 301) for y in range(-300, 301) if abs(x) + abs(y) <= 300]
print("số đích hợp lệ:", len(targets))

BFS toàn cục: 373321 ô, 0.83s
số đích hợp lệ: 180601


In [19]:
sol = Solution()
bad = [(p, REF[p]) for p in targets if formula(*p) != REF[p] or f_memo(*p) != REF[p]]
print("công thức + memo (cache dùng chung) sai ở", len(bad), "ô /", len(targets))
assert not bad
random.seed(1197)
for p in random.sample(targets, 2000):     # Solution tạo cache mới mỗi lần gọi -> thử mẫu
    assert sol.minKnightMoves(*p) == REF[p]
sample = random.sample(targets, 120) + [(0, 0), (1, 0), (2, 2), (1, 1), (-1, -1), (300, 0), (0, -300), (150, -150)]
for p in sample:
    for name, f in [("bfs_symmetric", bfs_symmetric), ("bfs_bidirectional", bfs_bidirectional)]:
        assert f(*p) == REF[p], (name, p)
near = [p for p in targets if max(abs(p[0]), abs(p[1])) <= 40]
for p in random.sample(near, 150):          # BFS thẳng chậm -> chỉ thử các đích gần
    assert bfs_plain(*p) == REF[p], p
print("BFS các kiểu khớp REF trên", len(sample), "đích (+150 đích gần cho BFS thẳng)")

công thức + memo (cache dùng chung) sai ở 0 ô / 180601


BFS các kiểu khớp REF trên 128 đích (+150 đích gần cho BFS thẳng)


In [20]:
# Tính chẵn lẻ + ví dụ đề + test thời gian worst case
assert all(REF[p] % 2 == (abs(p[0]) + abs(p[1])) % 2 for p in targets)
for p in [(150, 150), (300, 0), (-200, 100)]:
    t = time.perf_counter(); r = Solution().minKnightMoves(*p); dt = time.perf_counter() - t
    print(p, r, f"memo {dt * 1000:.2f}ms", "| bidir", bfs_bidirectional(*p))
print("test giới hạn OK")

(150, 150) 100 memo 0.81ms | bidir 100
(300, 0) 150 memo 0.73ms | bidir 150
(-200, 100) 100 memo 2.05ms | bidir 100
test giới hạn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chặn vùng quá chặt: chỉ cho tọa độ `≥ 0`

In [21]:
def wrong_nonneg(x, y):
    x, y = abs(x), abs(y)
    if (x, y) == (0, 0):
        return 0
    seen, q, steps = {(0, 0)}, deque([(0, 0)]), 0
    while q:
        steps += 1
        for _ in range(len(q)):
            cx, cy = q.popleft()
            for dx, dy in MOVES:
                p = (cx + dx, cy + dy)
                if p == (x, y):
                    return steps
                if 0 <= p[0] <= x + 2 and 0 <= p[1] <= y + 2 and p not in seen:   # SAI: >= 0
                    seen.add(p); q.append(p)
        if steps > 50:
            return "không tìm thấy"
    return "không tìm thấy"


for p in [(1, 1), (1, 0), (0, 1), (2, 2)]:
    print(p, wrong_nonneg(*p), "đúng là", REF[p])

(1, 1) 4 đúng là 2
(1, 0) 3 đúng là 3
(0, 1) 3 đúng là 3
(2, 2) 4 đúng là 4


**Vì sao sai:** `(1, 1)` chỉ đến được trong 2 bước qua `(2, -1)` hoặc `(-1, 2)` — đều có tọa độ âm. Chặn `≥ 0` làm đường dài hơn (4 bước). **Sửa:** cho phép `≥ -2` (hoặc chặn rộng hơn).

### Lỗi 2 — Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** hàng đợi

In [22]:
def wrong_mark_late(x, y, limit=200000):
    q = deque([(0, 0, 0)])
    seen = set()
    pushes = 0
    while q:
        cx, cy, d = q.popleft()
        if (cx, cy) == (x, y):
            return d, pushes
        if (cx, cy) in seen:
            continue
        seen.add((cx, cy))                     # đánh dấu muộn
        for dx, dy in MOVES:
            q.append((cx + dx, cy + dy, d + 1)); pushes += 1
            if pushes > limit:
                return "quá nhiều phần tử", pushes
    return None


def count_pushes_correct(x, y):
    seen, q, pushes = {(0, 0)}, deque([(0, 0, 0)]), 0
    while q:
        cx, cy, d = q.popleft()
        if (cx, cy) == (x, y):
            return d, pushes
        for dx, dy in MOVES:
            p = (cx + dx, cy + dy)
            if p not in seen:
                seen.add(p); q.append((*p, d + 1)); pushes += 1


print("đánh dấu muộn :", wrong_mark_late(6, 6))
print("đánh dấu sớm  :", count_pushes_correct(6, 6))

đánh dấu muộn : (4, 928)
đánh dấu sớm  : (4, 222)


**Vì sao tệ:** một ô có thể bị đưa vào hàng đợi nhiều lần (từ nhiều ô cha cùng lớp) → hàng đợi phình to gấp ~8 lần. Kết quả vẫn đúng nhưng tốn bộ nhớ/thời gian, dễ TLE/MLE. **Sửa:** `seen.add` ngay khi `append`.

### Lỗi 3 — Truy hồi thiếu trường hợp cơ sở `x + y == 2`

In [23]:
@lru_cache(maxsize=None)
def wrong_base(x, y):
    x, y = abs(x), abs(y)
    if x < y:
        x, y = y, x
    if x == 0 and y == 0:
        return 0
    if (x, y) in ((2, 1),):
        return 1
    if x + y > 40:
        return 10**6
    return 1 + min(wrong_base(x - 2, y - 1), wrong_base(x - 1, y - 2))


try:
    print(wrong_base(1, 1))
except RecursionError:
    print("RecursionError: (1,1) -> (1,0) -> (1,1) ... vòng lặp vô hạn")

RecursionError: (1,1) -> (1,0) -> (1,1) ... vòng lặp vô hạn


**Vì sao sai:** từ `(1,1)`, hai lựa chọn là `(|1−2|, |1−1|) = (1, 0)` và `(0, 1) → (1, 0)`; từ `(1,0)` lại về `(1, 1)` và `(0, 2)`... sinh **chu trình** — đệ quy không bao giờ dừng. Cơ sở `x + y == 2 → 2` cắt đứt chu trình. **Sửa:** thêm `if x + y == 2: return 2`.

### Lỗi 4 — BFS không chặn vùng (bàn vô hạn) với đích xa
Không sai về kết quả nhưng số ô thăm tăng theo `d^2` ở **cả 4 góc**: với `(150,150)` tốn hàng trăm nghìn ô. Hãy so sánh số ô đã thăm:

In [24]:
def visited_count_unbounded(x, y):
    seen, q = {(0, 0)}, deque([(0, 0)])
    while q:
        cx, cy = q.popleft()
        if (cx, cy) == (x, y):
            return len(seen)
        for dx, dy in MOVES:
            p = (cx + dx, cy + dy)
            if p not in seen:
                seen.add(p); q.append(p)


def visited_count_bounded(x, y):
    seen, q = {(0, 0)}, deque([(0, 0)])
    while q:
        cx, cy = q.popleft()
        if (cx, cy) == (x, y):
            return len(seen)
        for dx, dy in MOVES:
            p = (cx + dx, cy + dy)
            if -2 <= p[0] <= x + 2 and -2 <= p[1] <= y + 2 and p not in seen:
                seen.add(p); q.append(p)


print("không chặn:", visited_count_unbounded(60, 60), "| có chặn:", visited_count_bounded(60, 60))

không chặn: 22312 | có chặn: 4218


## 14. Mẹo & nhận dạng pattern

- **"Số bước ít nhất", mỗi bước chi phí như nhau** → BFS.
- **Không gian vô hạn** → tìm cách **chặn**: đối xứng, hình hộp bao quanh đích, hoặc heuristic (A*).
- **Biết cả nguồn lẫn đích** → BFS hai chiều giảm mạnh số trạng thái.
- **Bài lưới có cấu trúc hình học đều** → thử in bảng khoảng cách, tìm quy luật/truy hồi/công thức.
- **Kiểm tra chẵn lẻ** là cách debug nhanh cho bài quân mã.

## 15. Biến thể

1. **Bàn cờ hữu hạn `n × n`, từ ô A đến ô B** → BFS thường (chặn trong biên bàn).
2. **LeetCode 688 — Knight Probability in Chessboard:** xác suất còn trên bàn sau `k` bước → DP.
3. **LeetCode 935 — Knight Dialer:** đếm số dãy bấm phím theo nước mã → DP.
4. **Quân cờ khác (ví dụ "lạc đà" `(1,3)`)** → cùng BFS, đổi `MOVES`; chú ý có thể có ô không đến được (parity).
5. **Có ô cấm** → BFS + tập ô cấm; mất tính đối xứng nên không dùng công thức.

In [25]:
def bfs_general(target, moves, blocked=frozenset(), R=20):
    seen, q = {(0, 0)}, deque([((0, 0), 0)])
    while q:
        p, d = q.popleft()
        if p == target:
            return d
        for dx, dy in moves:
            nxt = (p[0] + dx, p[1] + dy)
            if max(abs(nxt[0]), abs(nxt[1])) <= R and nxt not in seen and nxt not in blocked:
                seen.add(nxt); q.append((nxt, d + 1))
    return -1


camel = [(a * 1, b * 3) for a in (1, -1) for b in (1, -1)] + [(a * 3, b * 1) for a in (1, -1) for b in (1, -1)]
print("mã tới (5,5):", bfs_general((5, 5), MOVES), "| lạc đà tới (1,0):", bfs_general((1, 0), camel), "| lạc đà tới (4,0):", bfs_general((4, 0), camel))
print("mã tới (1,1) khi cấm (2,-1) và (-1,2) ->", bfs_general((1, 1), MOVES, blocked={(2, -1), (-1, 2)}))

mã tới (5,5): 4 | lạc đà tới (1,0): -1 | lạc đà tới (4,0): 4
mã tới (1,1) khi cấm (2,-1) và (-1,2) -> 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS thẳng | Lan đều, chặn hình vuông | `O(R^2)` | `O(R^2)` | Oracle, hiểu bài |
| 2. BFS đối xứng | Góc phần tư 1, chặn `≥ -2` | `O(XY)` | `O(XY)` | Phỏng vấn, an toàn |
| 3. BFS hai chiều | Lan từ hai đầu | `O(XY)` (thực tế nhỏ) | `O(XY)` | Không gian lớn, biết đích |
| 4. DFS + memo | Truy hồi về gốc | nhỏ | nhỏ | Nộp bài, code ngắn |
| 5. Công thức | Toán học | `O(1)` | `O(1)` | Khi nhớ được và đã kiểm chứng |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `minKnightMoves(0, 2)`, `minKnightMoves(-3, 0)` = ?

**Bài 2 (tính tay):** dùng truy hồi `f(x,y) = 1 + min(f(x−2,y−1), f(x−1,y−2))` tính `f(4, 0)`.

**Bài 3 (điền TODO):** hoàn thiện `neighbors(x, y)` trả về 8 ô kề theo nước mã.

**Bài 4 (tự viết lại):** viết BFS trả về **đường đi** (danh sách ô) ngắn nhất, không chỉ độ dài.

**Bài 5 (mini challenge):** trên bàn hữu hạn `n × n` (ô `0..n-1`), tính số bước ít nhất từ `(0,0)` tới `(n-1, n-1)`, trả `-1` nếu không tới được (ví dụ `n = 2`).

In [26]:
def neighbors(x, y):
    # TODO: trả về list 8 tuple
    pass


def knight_path(x, y):
    # TODO: BFS + parent, trả về list các ô từ (0,0) tới (x,y)
    raise NotImplementedError


def knight_finite(n):
    # TODO
    raise NotImplementedError

# assert sorted(neighbors(0, 0)) == sorted(MOVES)
# assert knight_finite(2) == -1

## 18. Gợi ý 3 mức

- **Mức 1:** Ô kề là 8 vector `(±1,±2), (±2,±1)`. Bài toán là BFS.
- **Mức 2:** Đối xứng về góc phần tư thứ nhất; cho phép lệch âm tối đa 2 ô.
- **Mức 3:** Bài 4: lưu `parent[p] = ô trước` khi đưa `p` vào hàng đợi, rồi lần ngược từ đích. Bài 5: BFS trong biên `0 ≤ x, y < n`; `n = 1 → 0`, `n = 2 → -1`, `n = 3 → 4`.

## 19. Đáp án tham khảo

**Bài 1:** `(0, 2) → 2` (một đường: `(0,0) → (2,1) → (0,2)`), `(-3, 0) → 3` (đối xứng với `(3,0)`, ví dụ `(0,0) → (2,1) → (4,2) → (3,0)`; khớp chẵn lẻ: `3` lẻ).

**Bài 2:** `f(4,0) = 1 + min(f(2,1), f(3,2))`; `f(2,1) = 1 + min(f(0,0), f(1,1)) = 1 + 0 = 1` → `f(4,0) = 2` (`(0,0) → (2,1) → (4,0)`).

In [27]:
assert Solution().minKnightMoves(0, 2) == 2 and Solution().minKnightMoves(-3, 0) == 3
assert Solution().minKnightMoves(4, 0) == 2


def neighbors(x, y):
    return [(x + dx, y + dy) for dx, dy in MOVES]


def knight_path(x, y):
    tx, ty = x, y
    R = max(abs(x), abs(y)) + 4
    parent = {(0, 0): None}
    q = deque([(0, 0)])
    while q:
        p = q.popleft()
        if p == (tx, ty):
            break
        for nb in neighbors(*p):
            if max(abs(nb[0]), abs(nb[1])) <= R and nb not in parent:
                parent[nb] = p; q.append(nb)
    path, p = [], (tx, ty)
    while p is not None:
        path.append(p); p = parent[p]
    return path[::-1]


def knight_finite(n):
    target = (n - 1, n - 1)
    dist = {(0, 0): 0}
    q = deque([(0, 0)])
    while q:
        p = q.popleft()
        if p == target:
            return dist[p]
        for nb in neighbors(*p):
            if 0 <= nb[0] < n and 0 <= nb[1] < n and nb not in dist:
                dist[nb] = dist[p] + 1; q.append(nb)
    return -1


assert sorted(neighbors(0, 0)) == sorted(MOVES)
random.seed(19)
for p in random.sample(targets, 60) + [(0, 0), (1, 0), (5, 5)]:
    path = knight_path(*p)
    assert path[0] == (0, 0) and path[-1] == p and len(path) - 1 == REF[p]
    assert all((b[0] - a[0], b[1] - a[1]) in MOVES for a, b in zip(path, path[1:]))
assert [knight_finite(n) for n in (1, 2, 3, 4, 5)] == [0, -1, 4, 2, 4]
print(knight_path(5, 5), "| Đáp án bài tập: OK")

[(0, 0), (1, 2), (2, 4), (3, 6), (5, 5)] | Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
# BFS theo lớp trên lưới vô hạn có chặn
x, y = abs(x), abs(y)
seen, q, steps = {(0, 0)}, deque([(0, 0)]), 0
while q:
    steps += 1
    for _ in range(len(q)):
        cx, cy = q.popleft()
        for dx, dy in MOVES:
            p = (cx + dx, cy + dy)
            if p == (x, y): return steps
            if -2 <= p[0] <= x + 2 and -2 <= p[1] <= y + 2 and p not in seen:
                seen.add(p); q.append(p)

# DFS + memo
f(x,y): x,y=abs, x>=y; (0,0)->0; x+y==2 -> 2; else 1+min(f(x-2,y-1), f(x-1,y-2))
```

**Checklist:**
- [ ] `(0,0)` trả `0`; `(1,0)` trả `3`; `(1,1)` trả `2`; `(2,2)` trả `4`.
- [ ] Đã đưa về `|x|, |y|` (đối xứng).
- [ ] Vùng chặn cho phép tọa độ âm tới `-2`.
- [ ] `seen.add` ngay khi đưa vào hàng đợi.
- [ ] Đệ quy có đủ cơ sở (`x+y == 0` và `x+y == 2`) để không lặp vô hạn.
- [ ] Đối chiếu tính chẵn lẻ: đáp án `≡ x + y (mod 2)`.